# Does Pre-Treatment Morphology Predict Eventual Fate? (vem 1uM, 48hr and 144hr Pre-Treatment)

Same question, asked at two different pre-treatment durations (`config.CONDITIONS`): `48hr` (frame 1 is day -2, day 0 is frame 3) and `144hr` (frame 1 is day -6, day 0 is frame 7). Neither condition's frame-1 cells have a
single fate label: `cloneClassification` only classifies cells present
at day 0, and by then a frame-1 lineage can have already split
into several day-0 descendants with *different* fates. Each frame-1 cell
instead gets a **mixture**: the fraction of its day-0 descendants that
end up dying, persisting, as a slow proliferator, or as a fast
proliferator (`shared/lineage_fate.py`).

**Question**: does a cell's shape *before* treatment predict that
mixture? For each cell, this notebook predicts its fate mixture from the
mixtures of its k nearest neighbors *in real morphological distance*
(CAJAL's Gromov-Wasserstein distance matrix, not a UMAP/PHATE
projection, which distorts distances) and tests whether that prediction
is better than chance via permutation: repeatedly reassign which
mixture belongs to which cell at random, and see how often a random
reassignment predicts as well as the real one does. This is run
separately for each condition - they're two independent datasets, never
pooled into one CAJAL run or one test.

Requires `main.py` to have already been run for BOTH conditions (for
each condition's own `selected_metadata.csv` and
`{cell,nuc}_qgw_075P_dists.csv` under `output/{condition}/cajal_output/`),
and `config.raw_drug_dir()`/`lineage_mat_path()`/`clone_kmeans_mat_path()`
to point at the real raw data for each condition.

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# assumes Jupyter's cwd is this notebook's own directory
# (analyses/02_vem_pretreatment/notebooks/), so the analysis dir - and
# config.py - is one level up, and the repo root - where shared/ lives -
# is two more levels up from there
NOTEBOOK_DIR = os.getcwd()
ANALYSIS_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
REPO_ROOT    = os.path.abspath(os.path.join(ANALYSIS_DIR, '..', '..'))
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, ANALYSIS_DIR)
import config

# config.py's OUTPUT_DIR is relative to the analysis directory, not this
# notebook's own cwd - re-anchor it to an absolute path once, here.
# config.output_dir()/cajal_output_dir() both derive from this one
# variable, so re-anchoring it here is enough for every condition's own
# subtree too.
config.OUTPUT_DIR = os.path.join(ANALYSIS_DIR, config.OUTPUT_DIR)

from shared.cajal.embed import load_distance_matrix
from shared.load_data import match_labels_to_metadata
from shared.morphology_predicts_fate import knn_prediction_permutation_test, plot_permutation_test
from shared import lineage_fate

## Method

For each condition (`48hr`, `144hr`), and within it each modality (cell, nucleus):

1. **Load the real morphological distance matrix** (`load_distance_matrix`)
   and this condition's own selected-cell metadata, and match the two
   (`match_labels_to_metadata`) - no embedding involved, the distance
   matrix itself is the morphology representation.
2. **Resolve each frame-1 lineage's fate mixture** (`shared/lineage_fate.py`,
   same as before): `pct_dying`/`pct_persisting`/`pct_slow_prolif`/`pct_fast_prolif`
   per lineage, merged in by `lin_id`. A cell whose lineage dies out
   before day 0 has no mixture to predict or predict from, so it's
   dropped from both the metadata AND the distance matrix (dropping a
   row from one without the other would misalign every remaining pair).
3. **Predict each cell's mixture from its k nearest morphological
   neighbors**' own (real) mixtures, and test significance by comparing
   that prediction's error against the same prediction made after
   randomly reassigning mixtures to cells, many times
   (`knn_prediction_permutation_test` - see its docstring for the exact
   method).
4. **Plot**: one figure per (condition, modality) - the permutation null
   distribution with the real (observed) prediction error marked
   (`plot_permutation_test`), saved under that condition's own
   `output/{condition}/plots/01_morphology_predicts_fate/`.

In [ ]:
CATEGORY_NAMES = lineage_fate.SUBCLASSES
PCT_COLS = [f'pct_{c}' for c in CATEGORY_NAMES]
MODALITY_NAMES = {'cell': 'cell', 'nuc': 'nucleus'}

# number of nearest morphological neighbors averaged for each cell's
# prediction, and number of random label reassignments making up the
# permutation test's null distribution
K_NEIGHBORS     = 10
N_PERMUTATIONS  = 10_000
SEED            = 42


def process_modality(condition, modality, fate_pct):
    '''
    Load this condition+modality's real distance matrix and metadata,
    attach each cell's fate mixture, run the KNN prediction + permutation
    test, and plot the result.

    Inputs:
        condition: key into config.CONDITIONS ('48hr' or '144hr')
        modality:  'cell' or 'nuc'
        fate_pct:  from lineage_fate.compute_frame1_fate_percentages
    Outputs:
        observed_error, null_errors, p_value: see knn_prediction_permutation_test
    '''
    cajal_output_dir = config.cajal_output_dir(condition)
    plots_dir = os.path.join(config.output_dir(condition), 'plots', '01_morphology_predicts_fate')
    svg_dir   = os.path.join(plots_dir, 'svgs')
    os.makedirs(svg_dir, exist_ok=True)

    dist_csv = os.path.join(cajal_output_dir, f'{modality}_qgw_{config.N_POINTS:03d}P_dists.csv')
    labels, dist_matrix = load_distance_matrix(dist_csv)

    selected = pd.read_csv(os.path.join(cajal_output_dir, 'selected_metadata.csv'))
    df = match_labels_to_metadata(labels, selected)
    df = df.merge(fate_pct, on='lin_id', how='left')

    keep = df[PCT_COLS[0]].notna().to_numpy()
    n_dropped = (~keep).sum()
    if n_dropped > 0:
        day0_frame = config.CONDITIONS[condition]['day0_frame']
        print(f"  Dropping {n_dropped}/{len(df)} frame-1 cell(s) with no day-0 descendants in "
              f"lineage_frame{day0_frame}_to_frame1.mat (lineage died before day 0)")
    df = df[keep].reset_index(drop=True)
    dist_matrix = dist_matrix[keep][:, keep]  # same rows dropped on both axes, so pairs stay aligned

    mixture = df[PCT_COLS].to_numpy()
    print(f"[{condition}/{modality}] {len(df)} frame-1 cells with a resolved fate mixture")

    observed_error, null_errors, p_value, _ = knn_prediction_permutation_test(
        dist_matrix, mixture, k=K_NEIGHBORS, n_permutations=N_PERMUTATIONS, seed=SEED,
    )
    print(f"  observed mean prediction error = {observed_error:.4f} "
          f"(null mean = {null_errors.mean():.4f}, p = {p_value:.4f})")

    fig = plot_permutation_test(null_errors, observed_error, p_value, title=f'{MODALITY_NAMES[modality]} morphology')
    fig.savefig(os.path.join(plots_dir, f'{modality}.pdf'), bbox_inches='tight')
    fig.savefig(os.path.join(svg_dir, f'{modality}.svg'), bbox_inches='tight')
    plt.close(fig)
    print(f"  Saved to {plots_dir}")

    return observed_error, null_errors, p_value

## Run: both conditions, both modalities

In [ ]:
results = {}

for condition, info in config.CONDITIONS.items():
    print(f"=== {condition} ({info['drug']}) ===")
    raw_to_global = lineage_fate.build_raw_to_global_lin_id(config.raw_drug_dir(condition))
    fate_pct = lineage_fate.compute_frame1_fate_percentages(
        config.lineage_mat_path(condition), config.clone_kmeans_mat_path(condition), raw_to_global,
        dying_m_label=config.DYING_M_LABEL, persisting_m_label=config.PERSISTING_M_LABEL,
        prolif_m_label=config.PROLIF_M_LABEL,
        slow_k_labels=config.SLOW_K_LABELS, fast_k_labels=config.FAST_K_LABELS,
    )
    print(f"{len(fate_pct)} frame-1 lineages have at least one day-0 descendant")

    for modality in ['cell', 'nuc']:
        results[(condition, modality)] = process_modality(condition, modality, fate_pct)
    print()

## Export plot data

Combines both modalities' permutation-test results into one CSV per
condition (`plot_data.csv`, alongside the PDFs/SVGs above, under that
condition's own `output/{condition}/plots/01_morphology_predicts_fate/`):
one row per permutation, with `observed_error`/`p_value` repeated on
every row of their modality so the whole histogram - null distribution,
observed value, and p-value - can be redrawn from this one flat file.

In [ ]:
for condition in config.CONDITIONS:
    plot_data = pd.concat([
        pd.DataFrame({
            'modality': modality,
            'null_error': null_errors,
            'observed_error': observed_error,
            'p_value': p_value,
        })
        for modality, (observed_error, null_errors, p_value) in {
            m: results[(condition, m)] for m in ['cell', 'nuc']
        }.items()
    ], ignore_index=True)

    plots_dir = os.path.join(config.output_dir(condition), 'plots', '01_morphology_predicts_fate')
    plot_data_path = os.path.join(plots_dir, 'plot_data.csv')
    plot_data.to_csv(plot_data_path, index=False)
    print(f"[{condition}] Saved plot data ({len(plot_data)} rows) to {plot_data_path}")